# Term and Temp Hiring Outcomes

Are term and temp job announcements less likely to end in a selection than permanent ones, holding job type fixed?

Data: GSA's [Federal Hiring Assessments and Selection Outcome Dataset](https://catalog.data.gov/dataset/federal-hiring-assessments-and-selection-outcome-dataset), USAJOBS announcements from the 24 CFO Act agencies, opened FY2020 through Dec 31, 2023. The outcome is whether a selection was made off the announcement (a hiring manager picked someone; not necessarily an accepted offer).

GSA dropped temporary-only announcements from this dataset, so "temp" here means announcements listing Temporary as one of several appointment types.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyfixest as pf
import requests

pd.set_option("display.width", 200)
# Fixed-effect cells with a single listing carry no within-cell information; pyfixest drops them and warns each time.
warnings.filterwarnings("ignore", message=r".*singleton fixed effect\(s\) dropped")

In [2]:
BASE = "https://inventory.data.gov/dataset/4189591b-d78e-4ad3-9d75-582175ad0b3f/resource"
FILES = [
    f"{BASE}/ef9d8bb1-2b4f-4618-98d1-bf96deaa6503/download/hiring_selections_and_outcomes_public_data_set_as_of_dec_31_2023_fy2020_fy2021.xlsx",
    f"{BASE}/dc0a8624-0f45-4d39-97d7-2a6694810d19/download/hiring_selections_and_outcomes_public_data_set_as_of_dec_31_2023_fy2022_fy2023.xlsx",
    f"{BASE}/6de5e82a-3e16-491c-baf9-204c3b2f4767/download/hiring_selections_and_outcomes_public_data_set_as_of_dec_31_2023_fy2024.xlsx",
]
EXPECTED_ROWS = 1161779

# GSA counts an announcement toward selection outcomes only after it has been
# open for four months. The data runs through Dec 31, 2023.
LAST_MATURE_OPEN_DATE = "2023-08-31"

GROUPS = ["Permanent", "Term", "Multiple incl. Temporary"]

SPECS = {
    "raw": "{y} ~ term + temp_mix",
    "series x grade FE": "{y} ~ term + temp_mix | series_grade",
    "+ bureau, FY, ann. type FE": "{y} ~ term + temp_mix | series_grade + ab + fy + anntype",
    "series x grade x bureau FE + FY + ann. type": "{y} ~ term + temp_mix | sg_ab + fy + anntype",
    "previous + log(apps+1)": "{y} ~ term + temp_mix + np.log1p(apps) | sg_ab + fy + anntype",
}

## Download and load

Downloads the files into `gsa_data/` the first time.

In [3]:
def download(data_dir):
    """Download the GSA xlsx files that aren't already in data_dir."""
    data_dir.mkdir(exist_ok=True)
    for url in FILES:
        path = data_dir / url.rsplit("/", 1)[1]
        if path.exists():
            continue
        print(f"Downloading {path.name}")
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)


def load(data_dir):
    """Read and stack the yearly files."""
    frames = []
    for url in FILES:
        path = data_dir / url.rsplit("/", 1)[1]
        sheets = pd.read_excel(path, sheet_name=None, engine="calamine", dtype=str)
        if len(sheets) != 1:
            raise ValueError(f"Expected one sheet in {path.name}, found {list(sheets)}")
        frames.append(next(iter(sheets.values())))
    df = pd.concat(frames, ignore_index=True)
    if len(df) != EXPECTED_ROWS:
        raise ValueError(f"Expected {EXPECTED_ROWS} rows, got {len(df)}")
    return df

In [4]:
data_dir = Path("gsa_data")
download(data_dir)
raw = load(data_dir)
raw.shape

(1161779, 41)

## Prepare

GSA counts an announcement toward selection outcomes only after it has been open four months, so announcements opened after Aug 31, 2023 are dropped. Job type is occupational series x pay plan/grade; `sg_ab` adds agency/bureau.

In [5]:
def appointment_group(appt_type, appt_list):
    """Collapse Appointment_Type / Appointment_Type_List into analysis groups."""
    if appt_type == "Permanent":
        return "Permanent"
    if isinstance(appt_type, str) and appt_type.startswith("Term"):
        return "Term"
    if appt_type == "Multiple Appointment Types":
        parts = [p.strip() for p in str(appt_list).split(",")]
        if "Temporary" in parts:
            return "Multiple incl. Temporary"
        if "Term" in parts:
            return "Multiple incl. Term (no temp)"
        return "Multiple other"
    return "Other (intermittent/seasonal/etc.)"


def prepare(df):
    """Drop immature announcements and build outcome, group, and fixed-effect columns."""
    df = df.copy()
    df["open"] = pd.to_datetime(df.Announcement_Open_Date)
    df = df[df.open <= LAST_MATURE_OPEN_DATE].copy()
    df["appt"] = [appointment_group(t, l) for t, l in zip(df.Appointment_Type, df.Appointment_Type_List)]
    df["sel"] = (df.Selection_Made_off_this_Job_Announcement == "Yes").astype(int)
    df["sel_any"] = ((df.sel == 1) | (df.Selection_Made_through_Separate_Announcement == "Yes")).astype(int)
    df["series_grade"] = df.Job_Series_Number.str.strip("'") + "|" + df.Pay_Plan + "-" + df.Grade.str.strip("'")
    df["ab"] = df.Agency + "|" + df.Bureau.fillna("(none)")
    df["sg_ab"] = df.series_grade + "|" + df.ab
    df["fy"] = df.Announcement_Open_Date_FY
    df["anntype"] = df.Announcement_Type.str.upper().str.strip()
    df["apps"] = pd.to_numeric(df.Total_Applications, errors="coerce")
    df["term"] = (df.appt == "Term").astype(int)
    df["temp_mix"] = (df.appt == "Multiple incl. Temporary").astype(int)
    return df

In [6]:
df = prepare(raw)
len(df)

1061305

## Raw selection rates

`sel_any` also counts a selection made off a separate announcement for the same position (agencies often post public and internal versions at once). Only USA Staffing reports it.

In [7]:
def raw_rates(df):
    """Selection rates by appointment group."""
    return (
        df.groupby("appt")
        .agg(n=("sel", "size"), sel_rate=("sel", "mean"), sel_any_rate=("sel_any", "mean"), median_apps=("apps", "median"))
        .sort_values("n", ascending=False)
    )


raw_rates(df).round(3)

,n,sel_rate,sel_any_rate,median_apps
appt,,,,
Permanent,1005987,0.568,0.674,16.0
Term,33716,0.503,0.592,16.0
Other (intermittent/seasonal/etc.),9553,0.504,0.605,8.0
Multiple incl. Temporary,5903,0.443,0.548,7.0
Multiple other,3816,0.631,0.721,19.0
Multiple incl. Term (no temp),2330,0.456,0.594,9.0


## Holding job type fixed

Linear probability models, standard errors clustered by bureau. `term` and `temp_mix` are percentage-point differences from permanent. The last spec adds log applications, which is likely part of the mechanism rather than a confounder.

In [8]:
def regressions(df):
    """Linear probability models of selection on term / temp, clustered by bureau."""
    m = df[df.appt.isin(GROUPS)]
    rows = []
    # sel_any uses Selection_Made_through_Separate_Announcement, which only USA Staffing reports
    for y, sample in [("sel", m), ("sel_any", m[m.Source == "USA_Staffing"])]:
        for name, f in SPECS.items():
            fit = pf.feols(f.format(y=y), data=sample, vcov={"CRV1": "ab"})
            t = fit.tidy()
            for v in ["term", "temp_mix"]:
                rows.append(dict(outcome=y, spec=name, var=v, coef=t.loc[v, "Estimate"], se=t.loc[v, "Std. Error"], n=fit._N))
    res = pd.DataFrame(rows)
    res["ci_lo"] = res.coef - 1.96 * res.se
    res["ci_hi"] = res.coef + 1.96 * res.se
    return res


regressions(df).round(4)

,outcome,spec,var,coef,se,n,ci_lo,ci_hi
0,sel,raw,term,-0.0658,0.0242,1045606,-0.1132,-0.0184
1,sel,raw,temp_mix,-0.1256,0.0326,1045606,-0.1894,-0.0617
2,sel,series x grade FE,term,-0.0940,0.0148,1039573,-0.1230,-0.0650
3,sel,series x grade FE,temp_mix,-0.1006,0.0294,1039573,-0.1581,-0.0430
4,sel,"+ bureau, FY, ann. type FE",term,-0.0931,0.0174,1039363,-0.1271,-0.0591
5,sel,"+ bureau, FY, ann. type FE",temp_mix,-0.1163,0.0313,1039363,-0.1776,-0.0549
6,sel,series x grade x bureau FE + FY + ann. type,term,-0.1067,0.0129,1028408,-0.1319,-0.0815
7,sel,series x grade x bureau FE + FY + ann. type,temp_mix,-0.0953,0.0394,1028408,-0.1725,-0.0181
8,sel,previous + log(apps+1),term,-0.0607,0.0110,1026339,-0.0823,-0.0392
9,sel,previous + log(apps+1),temp_mix,-0.0542,0.0173,1026339,-0.0881,-0.0202


## Matched check

Within series x grade x bureau x FY cells that have both term and permanent listings, weighted by term listings.

In [9]:
def matched_cells(df):
    """Term-weighted term-minus-permanent gap within series x grade x bureau x FY cells that have both."""
    c = df[df.appt.isin(["Permanent", "Term"])]
    cell = c.groupby(["sg_ab", "fy", "appt"]).sel.agg(["mean", "size"]).unstack("appt").dropna()
    w = cell[("size", "Term")]
    return {
        "cells": len(cell),
        "term_listings_matched": int(w.sum()),
        "term_listings_total": int((c.appt == "Term").sum()),
        "perm_rate": float(np.average(cell[("mean", "Permanent")], weights=w)),
        "term_rate": float(np.average(cell[("mean", "Term")], weights=w)),
    }


matched_cells(df)

{'cells': 7501,
 'term_listings_matched': 30279,
 'term_listings_total': 33716,
 'perm_rate': 0.6150223093379188,
 'term_rate': 0.49436903464447307}

## By agency

Defense posts most term listings, so the overall result is mostly a DoD story.

In [10]:
def by_agency(df, top_n=8):
    """Term gap within each of the agencies posting the most term listings."""
    m = df[df.appt.isin(GROUPS)]
    rows = []
    for agency, n_term in m[m.term == 1].Agency.value_counts().head(top_n).items():
        sub = m[m.Agency == agency]
        t = pf.feols("sel ~ term + temp_mix | sg_ab + fy + anntype", data=sub, vcov={"CRV1": "ab"}).tidy()
        rows.append(dict(agency=agency, term_listings=n_term, coef=t.loc["term", "Estimate"], se=t.loc["term", "Std. Error"]))
    return pd.DataFrame(rows)


by_agency(df).round(4)

,agency,term_listings,coef,se
0,Department of Defense,24318,-0.1269,0.0114
1,Department of Veterans Affairs,2905,-0.1138,0.0077
2,Department of the Interior,1867,0.1447,0.0493
3,Department of Agriculture,1196,-0.0004,0.0187
4,National Aeronautics and Space Administration,853,-0.1431,0.0221
5,Department of Health and Human Services,781,-0.0135,0.0280
6,Department of Commerce,690,0.0272,0.0191
7,Department of Homeland Security,288,0.1187,0.1122


## DoD: how much is applicant pool size?

Science/tech = series 04xx (bio sciences), 08xx (engineering), 13xx (physical sciences), 15xx (math/stats/CS), 2210 (IT).

In [11]:
def is_stem(series):
    """Science/tech series: 04xx bio sciences, 08xx engineering, 13xx physical sciences, 15xx math/stats/CS, 2210 IT."""
    s = series.str.strip("'")
    return s.str[:2].isin(["04", "08", "13", "15"]) | (s == "2210")


def applicant_share(df, agency="Department of Defense"):
    """Term gap at one agency with and without controlling for log applications, all jobs vs STEM."""
    m = df[df.appt.isin(GROUPS) & (df.Agency == agency)]
    rows = []
    for label, sub in [("all jobs", m), ("STEM", m[is_stem(m.Job_Series_Number)])]:
        base = pf.feols("sel ~ term + temp_mix | sg_ab + fy + anntype", data=sub, vcov={"CRV1": "ab"}).tidy()
        ctrl = pf.feols("sel ~ term + temp_mix + np.log1p(apps) | sg_ab + fy + anntype", data=sub, vcov={"CRV1": "ab"}).tidy()
        terms = sub[sub.term == 1]
        perms = sub[(sub.term == 0) & (sub.temp_mix == 0)]
        rows.append(dict(
            jobs=label,
            term_listings=len(terms),
            median_apps_term=terms.apps.median(),
            median_apps_perm=perms.apps.median(),
            gap=base.loc["term", "Estimate"],
            gap_se=base.loc["term", "Std. Error"],
            gap_ctrl_apps=ctrl.loc["term", "Estimate"],
            gap_ctrl_apps_se=ctrl.loc["term", "Std. Error"],
        ))
    res = pd.DataFrame(rows)
    res["share_explained_by_apps"] = 1 - res.gap_ctrl_apps / res.gap
    return res


applicant_share(df).round(4)

,jobs,term_listings,median_apps_term,median_apps_perm,gap,gap_se,gap_ctrl_apps,gap_ctrl_apps_se,share_explained_by_apps
0,all jobs,24318,14.0,19.0,-0.1269,0.0114,-0.0663,0.0096,0.4776
1,STEM,3644,10.0,14.0,-0.1460,0.0122,-0.0682,0.0185,0.5332
